# LLMOps Capstone— Ship a Production Pipeline

**Module 14 · Lesson 14.6**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Whole Game — One Request, Six Stages
- The Router — Two Deterministic Decisions
- The Registry — Versioned Prompts, Cacheable Prefix
- Instrumentation — No Trace, No LLMOps
- The Eval Harness — Golden Set + Judge
- The CI Gate — Block the Bad Merge

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

## Naming LLMOps and shipping it are different skills

> 💡 **Info**
>
> Most engineers can say “golden-set eval” and “prompt caching” in an interview. Vanishingly few have actually wired a **CI gate that blocked their own pull request** for a quality regression. That gap — between describing LLMOps and doing it — is the whole point of this **integrative capstone project**. You are going to build one production-grade feature, a **QuickBite food-delivery review classifier** (five categories: refund, delivery, food_quality, service, other), and ship it behind every practice from this module: every call instrumented (14.1), two prompt versions A/B-routed (14.2), a hand-labeled golden set graded by an LLM-as-judge and gated in CI (14.3), and model routing plus prompt caching for cost (14.5). The classifier is the easy part. The **wiring** — the router that picks a version and a model, the trace that logs everything, the gate that blocks a bad merge — is the part that makes you employable at the senior level. This lesson is that wiring, one subsystem per step, each as runnable reference code.

### 🎯 What you will be able to do after this lesson

- **Create** one operated `POST /classify` feature by wiring the five Module 14 subsystems into a single request path and a single CI path.

- **Apply** the reference implementation of each glue piece — the router, the registry render, the trace record, the eval runner, the CI gate, the cost projection.

- **Analyze** which wiring decision each subsystem depends on (deterministic bucketing, a cacheable prefix, a different-family judge, a three-check gate) and what silently breaks without it.

- **Evaluate** your own repo against a senior-engineer rubric and a go-live checklist before you ship and open-source it.

> 📦 **Info**
>
> ✅ Before you startThis is the denouement of Module 14 — it introduces no new mechanism, it *connects* the ones you built. In **14.1** you instrumented a call; here every request emits that trace. In **14.2** you versioned prompts and bucketed an A/B; here the router does both per request. In **14.3** you built a golden set, a judge, and a CI gate; here they gate your own merges. In **14.5** you routed by cost and cached a prefix; here both run in the request path. The FastAPI and Streamlit app skills come from Modules 7 and 8. If a mechanism feels new, its lesson is the reference.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🍽️ **Analogy**
>
> Building this capstone is like **running a restaurant pass, not cooking one dish**. Anyone can cook a plate of food — that is the classifier. Running the *pass* is the hard part: every order gets routed to the right station, timed, checked against the ticket, and logged, and nothing leaves the kitchen without passing the expo. The LLMOps layer is that pass. **Where it breaks down:** a restaurant pass is judged in the moment, while your pipeline is judged by whether a stranger can open the repo cold and see the discipline in five minutes — so the wiring has to be legible, not just working.

> ℹ️ **Info**
>
> 📦 The artifact you are buildingA GitHub repo with code, prompts, a hand-labeled eval set, a CI workflow, a README, and a deployed demo. Lay it out this way and the rest writes itself:
> quickbite-classifier-llmops/
> ├── README.md                    # architecture + how to run + eval results + cost analysis
> ├── .env.example                 # ANTHROPIC_API_KEY, LANGFUSE_* (.env is gitignored)
> ├── prompts/
> │   ├── quickbite-classifier-v1.0.0.yaml   # baseline
> │   ├── quickbite-classifier-v1.1.0.yaml   # + 3 few-shot examples
> │   └── llm-judge-v1.yaml
> ├── eval/
> │   ├── golden/quickbite-classifier.jsonl  # 50+ hand-labeled rows
> │   ├── run_golden.py            # runs the set against a prompt version
> │   ├── judge.py                 # multi-dim LLM-as-judge
> │   └── gate.py                  # CI gate: floor + regression + cost
> ├── src/
> │   ├── registry.py              # load_prompt, pick_version   (14.2)
> │   ├── router.py                # pick_model by length        (14.5)
> │   ├── instrumented.py          # Langfuse-wrapped client     (14.3)
> │   ├── classifier.py            # glue: registry+router+instrumented+parse
> │   └── api.py                   # FastAPI - POST /classify
> ├── dashboard/app.py             # Streamlit dashboard
> └── .github/workflows/llm-eval.yml   # runs the gate on every PR to prompts/
> Take about one to two weeks of focused evening work, or one weekend at sprint pace. Build it in phase order — bootstrap the request path first, the eval and gate next, the dashboard last — because building the dashboard first means decorating a pipeline that does not exist yet.

> 📦 **Info**
>
> 🚫 Two misconceptions this capstone kills
> - **“If I can name the practices, I can ship them.”** The vocabulary is free; the wiring is the skill. A repo where a CI gate actually blocked a regression is worth a hundred that can only describe one.
> - **“An LLM-synthesized golden set is fine.”** When your judge and your producer share a model family and a writing style, the judge over-rewards the synthesized cases and the pass rate looks fake-high — worthless as a regression signal. Hand-label real reviews — and, since the golden set is committed to a public repo, anonymize any personal data (names, addresses, phone numbers, order IDs) before it ships.

> 💡 **Info**
>
> ⚠️ Anti-pattern — the four quiet failuresThe capstone rarely fails loudly; it fails in four quiet ways. **Building the dashboard first** (a beautiful UI on a non-existent pipeline, then weeks of rework). **A CI gate never tested firing** (you wrote it but never opened a deliberately-bad PR, so the first real regression slips through). **The same model as judge and producer** (self-preference bias; the judge keeps approving its own family). And **committing API keys** (a scanning bot pulls them within hours). Every step below is wired to prevent one of these.

---

## 🎯 Concept 1: The Whole Game — One Request, Six Stages

### The Whole Game — One Request, Six Stages

A POST /classify request threads through six wired stages - A/B pick, load from registry, model route, render + cache, instrumented call, parse - and returns the JSON contract. This is the entire capstone in miniature: wire these six into one endpoint that LOGS every request and GATES every prompt change.

Start by seeing the whole thing at once. A single request to `POST /classify` is not one API call — it is a small pipeline of six decisions and one logged outcome. The router picks a **prompt version** (the A/B decision, by `user_id`) and a **model** (the cost decision, by review length); the registry **loads** that version; the renderer builds a **cacheable prefix** plus the variable review; the **instrumented call** hits the model and writes a trace; and the result is **parsed** into the JSON contract `{category, confidence, prompt_version, model}`. That is the request path. The other half of the capstone is the CI path — every pull request that touches a prompt runs the golden set and is gated — which the later steps build. Master the wiring of these two paths and you have shipped production LLMOps. The block runs the six-stage `classify()` for two different requests, keyless; the illustrative artifact is the real FastAPI endpoint.

> 🍽️ **Analogy**
>
> The request path is **a food order crossing the pass**. The ticket comes in and the expediter routes it: which station cooks it (the model), which version of the recipe is live tonight (the prompt version), plating and garnish (the render), then a final check and a log at the pass before it leaves (the instrumented call and parse). Every order takes the same route through the same stations; what changes per order is which station and which recipe the expediter picks. Your `classify()` is the expediter, and the six stages are the stations every ticket crosses.

A short refund review comes from a user who buckets to the champion, and a long delivery review comes from a user who buckets to the candidate. What differs between the two responses?

**📝 `01_classify_whole_game.py`** - *runnable*

In [ ]:
# THE WHOLE GAME: one POST /classify request threads through SIX wired stages and returns the JSON contract.
# This is the entire capstone in miniature - route -> registry -> model pick -> render+cache -> instrumented call -> parse.
import hashlib
PROMPTS = {"v1.0.0": "baseline", "v1.1.0": "few-shot"}          # the prompt registry (from Lesson 14.2)
def bucket(user_id, candidate_pct=10):                          # (1) deterministic A/B pick by user_id
    return "v1.1.0" if int(hashlib.md5(user_id.encode()).hexdigest(), 16) % 100 < candidate_pct else "v1.0.0"
def pick_model(review):                                         # (3) cost routing by length (from Lesson 14.5)
    return "claude-haiku-4-5" if len(review) < 100 else "claude-sonnet-4-6"
def classify(review, user_id):
    version = bucket(user_id)                                   # (1) which prompt version
    prompt = PROMPTS[version]                                   # (2) load it from the registry
    model = pick_model(review)                                  # (3) which model
    # (4) render template + cache_control on the prefix, (5) instrumented Anthropic call, (6) parse - simulated here:
    category, confidence = ("refund", 0.94) if "refund" in review.lower() else ("delivery", 0.88)
    return {"category": category, "confidence": confidence, "prompt_version": version, "model": model}
print("short refund review ->", classify("Refund never arrived, awful.", "user_0001"))                    # champion + Haiku
print("long delivery review->", classify("The driver left it at the wrong building entirely and it took another full hour with support to sort out the mix-up.", "user_0017"))  # candidate + Sonnet
print()
print("Six stages, every request: A/B pick -> registry -> model route -> render+cache -> instrumented call -> parse.")
print("The whole capstone is wiring these six into one endpoint that LOGS everything and GATES every prompt change.")

# Output:
# short refund review -> {'category': 'refund', 'confidence': 0.94, 'prompt_version': 'v1.0.0', 'model': 'claude-haiku-4-5'}
# long delivery review-> {'category': 'delivery', 'confidence': 0.88, 'prompt_version': 'v1.1.0', 'model': 'claude-sonnet-4-6'}
#
# Six stages, every request: A/B pick -> registry -> model route -> render+cache -> instrumented call -> parse.
# The whole capstone is wiring these six into one endpoint that LOGS everything and GATES every prompt change.

**📝 `api.py`** - *illustrative*

In [ ]:
# THE GLUE: a real POST /classify endpoint wiring the subsystems - an illustrative minimal example (needs the stack + a key, not run here).
import os
from fastapi import FastAPI
from pydantic import BaseModel
from anthropic import Anthropic
from src.registry import load_prompt, pick_version     # deterministic A/B + versioned YAML (Lesson 14.2)
from src.router import pick_model                       # length-based model routing (Lesson 14.5)
from src.instrumented import instrumented_call          # Langfuse-wrapped, cache_control on the prefix (Lessons 14.3 + 14.5)

app = FastAPI()
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"], timeout=30)

class Req(BaseModel):
    review: str
    user_id: str

@app.post("/classify")
def classify(req: Req):
    version = pick_version(req.user_id)                          # (1) deterministic A/B bucket
    prompt = load_prompt("quickbite-classifier", version)       # (2) load the versioned prompt
    model = pick_model(req.review)                              # (3) route by review length
    resp = instrumented_call(                                   # (4)+(5) render + cache_control, traced call
        client, model=model, prompt=prompt, review=req.review,
        tags={"prompt_version": version, "user_id": req.user_id})
    category, confidence = parse(resp)                          # (6) parse the JSON output
    return {"category": category, "confidence": confidence, "prompt_version": version, "model": model}
# Output: (illustrative - needs FastAPI + the Anthropic SDK + Langfuse + a key; the real wired endpoint, not run here.)

- A request threads through **six stages**: A/B pick, registry load, model route, render + cache, instrumented call, parse.
- The short champion request returns `v1.0.0` + Haiku; the long candidate request returns `v1.1.0` + Sonnet — same path, different routing.
- The JSON contract — category, confidence, prompt_version, model — is what the endpoint returns and what the dashboard reads.
- The **illustrative artifact** is the real FastAPI endpoint wiring the same six stages with the actual registry, router, and instrumented client.

#### 💡 What just happened

⚡ What just happened?One request is six wired stages plus one logged outcome, returning a fixed JSON contract; the router changes the version and model per request. Tradeoff: more moving parts than a bare call, in exchange for a feature you can log, A/B, and gate. Next: the router that makes those two decisions.

- A request lights up six wired stages in sequence and emits the JSON contract
- A short-champion / long-candidate toggle changes the version and model

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Router — Two Deterministic Decisions

### The Router — Two Deterministic Decisions

Per request the router makes two calls: which prompt VERSION (A/B bucket by a hash of user_id, 90/10) and which MODEL (by review length). Deterministic bucketing with hashlib - not the salted built-in hash() - keeps A/B metrics clean, because the same user never flips versions between requests.

The router is two small functions, and each has a trap. **Version routing** assigns a user to the champion or the candidate by hashing their `user_id` and taking it modulo one hundred: below ten goes to the candidate, giving a ninety-ten split. The trap is *determinism*: if you used a random draw per request, the same user would see the candidate on one call and the champion on the next, and your A/B metrics would be mush. So you hash — and you use `hashlib`, because Python’s built-in `hash()` is salted per process and would bucket the same user differently after a restart. **Model routing** sends short reviews to a cheap model (Haiku) and long ones to a stronger model (Sonnet), on the length lever from Lesson 14.5 — and you could point the cheap lane at a budget model from Google or OpenAI just as easily. The block runs both decisions over a thousand users, keyless, and shows the split and the determinism.

> 🍴 **Analogy**
>
> The router is a **maitre d’ with a fixed seating rule**. A good host does not seat you at a random table each visit — regulars get the same section every time, so the waitstaff know their history and the service data stays coherent. That is deterministic bucketing: the same guest, the same lane, every visit, so the experiment’s books stay clean. The second decision — a quick two-top to the fast station, a big party to the full kitchen — is the length-based model routing: cheap traffic to the cheap model, hard traffic to the strong one.

You bucket users by a random draw on each request instead of a hash of user_id. What happens to your A/B experiment?

**📝 `02_router.py`** - *runnable*

In [ ]:
# THE ROUTER: two deterministic decisions per request - which prompt VERSION (A/B) and which MODEL (cost routing).
import hashlib
def bucket(user_id, candidate_pct=10):
    return "candidate v1.1.0" if int(hashlib.md5(user_id.encode()).hexdigest(), 16) % 100 < candidate_pct else "champion v1.0.0"
def pick_model(review):
    return "claude-haiku-4-5" if len(review) < 100 else "claude-sonnet-4-6"
users = ["user_{:04d}".format(i) for i in range(1000)]
cand = sum(1 for u in users if bucket(u).startswith("candidate"))
print("A/B split over {} users: {} champion, {} candidate  (~{:.0%} routed to the candidate).".format(
      len(users), len(users) - cand, cand, cand / len(users)))
same = [bucket("user_0042") for _ in range(3)]                 # DETERMINISTIC: same user, same bucket, every time
print("user_0042 asked 3x -> {}  (never flips - so A/B metrics stay clean).".format(", ".join(same)))
short, long = "Cold fries.", "The driver left it at the wrong building entirely and it took another full hour with support to sort out the mix-up."
print("model route: {!r} ({} chars) -> {}".format(short, len(short), pick_model(short)))
print("model route: long review ({} chars) -> {}".format(len(long), pick_model(long)))
print()
print("Deterministic bucketing keeps A/B clean (a user never switches versions); length routing sends cheap traffic to Haiku.")

# Output:
# A/B split over 1000 users: 899 champion, 101 candidate  (~10% routed to the candidate).
# user_0042 asked 3x -> champion v1.0.0, champion v1.0.0, champion v1.0.0  (never flips - so A/B metrics stay clean).
# model route: 'Cold fries.' (11 chars) -> claude-haiku-4-5
# model route: long review (116 chars) -> claude-sonnet-4-6
#
# Deterministic bucketing keeps A/B clean (a user never switches versions); length routing sends cheap traffic to Haiku.

- Version routing hashes `user_id`: `md5 % 100 < 10` goes to the candidate — a ninety-ten split (about one in ten users).
- It is **deterministic**: the same user asked three times lands in the same lane every time, so A/B metrics stay clean.
- Model routing sends the short review to Haiku and the long one to Sonnet on a length cutoff — the cost lever from 14.5.
- Use `hashlib`, not the built-in `hash()` — the latter is salted per process and would re-bucket users after a restart.

#### 💡 What just happened

⚡ What just happened?The router makes two deterministic decisions per request - version by a hashlib bucket of user_id (90/10), model by review length - and determinism is what keeps the A/B honest. Tradeoff: you must remember hashlib over the salted built-in hash(). Next: the registry that stores and renders the versions.

- 1000 users hashed into champion and candidate lanes at a 90/10 split
- A length dial that picks Haiku for short reviews and Sonnet for long

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The Registry — Versioned Prompts, Cacheable Prefix

### The Registry — Versioned Prompts, Cacheable Prefix

Prompts live in versioned YAML files, rendered into a STABLE prefix (system + few-shot) plus a VARIABLE suffix (the review). The stable prefix is marked cacheable and the version tags every trace. v1.0.0 is the baseline; v1.1.0 adds three few-shot examples targeting the exact failure modes the golden set covers.

The registry is where prompt versioning becomes real files. Each version is a YAML document — `quickbite-classifier-v1.0.0.yaml`, `-v1.1.0.yaml` — and `render()` assembles the message from it: a **stable prefix** (the system instruction plus any few-shot examples) followed by the **variable suffix** (the actual review). Two wiring details matter. First, the prefix is *identical on every call*, so you mark it cacheable with `cache_control` (Lesson 14.5) and only pay full price for the short, varying review — keep the review last or you bust the cache. Second, the **version tags every trace**, so the dashboard can split cost and pass-rate by version. The candidate `v1.1.0` adds three few-shot examples chosen to fix the exact cases the golden set is hardest on — an ambiguous one, a Spanish one, a multi-issue one. The block renders both versions, keyless, and shows the cacheable prefix growing while the suffix stays the review.

> 📖 **Analogy**
>
> The registry is a **recipe binder with dated cards**. A serious kitchen does not keep the recipe in someone’s head — every version is a dated card in a binder, so you know exactly which one was live when a dish went out, and you can roll back to last week’s card in seconds. The stable part of the card — the base method and the standing notes — is the same every service (your cacheable prefix); only tonight’s special order fills in the blank (the review). The version stamped on every ticket is how you later tell which card produced which result.

In the rendered message, why is the system-plus-few-shot block placed before the review and marked cacheable, with the review appended last?

**📝 `03_registry.py`** - *runnable*

In [ ]:
# THE PROMPT REGISTRY: versioned prompts loaded from YAML, rendered with the review, the stable prefix marked cacheable.
SYSTEM = "Classify a food-delivery review into exactly one of: refund, delivery, food_quality, service, other."
REGISTRY = {                                                   # loaded from prompts/quickbite-classifier-v{ver}.yaml
    "v1.0.0": {"system": SYSTEM, "few_shot": []},              # baseline, no examples
    "v1.1.0": {"system": SYSTEM, "few_shot": [                 # +3 few-shot: ambiguous, Spanish, multi-issue
        "'Money never came back' -> refund", "'Llego frio' -> food_quality", "'Late AND cold' -> delivery"]},
}
def render(version, review):
    p = REGISTRY[version]
    prefix = p["system"] + ("".join("\n" + ex for ex in p["few_shot"]))   # STABLE across calls -> cache_control here
    return {"prompt_version": version, "cacheable_prefix": prefix, "variable_suffix": "Review: " + review}
msg = render("v1.1.0", "Refund never processed.")
print("prompt_version:", msg["prompt_version"], "| prefix", len(msg["cacheable_prefix"]), "chars (cache_control=ephemeral, ~5-min TTL)")
print("cacheable prefix:", msg["cacheable_prefix"].replace("\n", "  |  "))
print("variable suffix (never cached):", repr(msg["variable_suffix"]))
print("v1.0.0 prefix", len(render("v1.0.0", "x")["cacheable_prefix"]), "chars (no few-shot) vs v1.1.0", len(msg["cacheable_prefix"]), "chars (+3 examples).")
print()
print("The system + few-shot block is identical every call, so it caches; only the review varies. The version tags every trace.")

# Output:
# prompt_version: v1.1.0 | prefix 191 chars (cache_control=ephemeral, ~5-min TTL)
# cacheable prefix: Classify a food-delivery review into exactly one of: refund, delivery, food_quality, service, other.  |  'Money never came back' -> refund  |  'Llego frio' -> food_quality  |  'Late AND cold' -> delivery
# variable suffix (never cached): 'Review: Refund never processed.'
# v1.0.0 prefix 100 chars (no few-shot) vs v1.1.0 191 chars (+3 examples).
#
# The system + few-shot block is identical every call, so it caches; only the review varies. The version tags every trace.

- Each version is loaded from a versioned YAML file; `v1.0.0` is the baseline, `v1.1.0` adds three few-shot examples.
- `render()` builds a **stable prefix** (system + few-shot) plus a **variable suffix** (`Review: ...`).
- The prefix is identical every call, so it is marked cacheable (`cache_control`, ephemeral ~five-minute window); only the review varies.
- The `prompt_version` tags every trace, so the dashboard can split cost and pass-rate by version.

#### 💡 What just happened

⚡ What just happened?Prompts live in versioned YAML; render() builds a cacheable stable prefix plus a variable review, and the version tags every trace. Tradeoff: a little file plumbing, in exchange for rollback, A/B, and caching in one place. Next: the trace that every downstream reader depends on.

- Two version cards - v1.0.0 baseline and v1.1.0 with three few-shot examples
- A render that highlights the cacheable prefix and the variable review

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Instrumentation — No Trace, No LLMOps

### Instrumentation — No Trace, No LLMOps

Every call emits ONE trace record with the field set the dashboard reads - the OpenTelemetry GenAI conventions from 14.1 (gen_ai.request.model, gen_ai.usage.*). Twelve required fields, a cost computed from the token counts. The dashboard, the A/B metrics, the cost report, and drift detection are ALL just readers of this one record.

Instrumentation is the foundation everything else stands on, which is why skipping it “to ship faster” is the most expensive shortcut in the capstone. Every call writes a single **trace record** — a flat object with a fixed set of fields following the **OpenTelemetry GenAI conventions** you met in Lesson 14.1: the model, the prompt version, the user, the input and output token counts, the computed cost, the latency, the parsed category and confidence, the cache-read tokens, and the environment. Twelve required fields in this build. The reason it matters is downstream: the **dashboard**, the **A/B metrics**, the **cost report**, and **drift detection** are all just *readers* of this one record. Drop a field and one of those readers silently breaks. In production the store is Langfuse (v3) or Arize Phoenix; here you build and validate the record itself. The block constructs the trace for a Haiku call, checks all twelve fields are present, and computes the cost, keyless.

> ✈️ **Analogy**
>
> The trace record is an aircraft’s **flight recorder**. Nobody looks at it while the flight goes well — but every investigation, every maintenance report, and every performance review reads the exact same recorded data afterward. If a field was never recorded, no downstream analysis can recover it. Your trace is that recorder: the dashboard, the A/B report, the cost accounting, and the drift monitor all replay the one record, so a missing field is a question none of them can ever answer.

You ship the classifier but skip writing the trace record to move faster. Which of these still works?

**📝 `04_instrumentation.py`** - *runnable*

In [ ]:
# INSTRUMENTATION: every call emits ONE trace record with the field set the dashboard reads (OpenTelemetry GenAI conventions, 14.1).
REQUIRED = ["trace_id", "gen_ai.request.model", "prompt_version", "user_id", "gen_ai.usage.input_tokens",
            "gen_ai.usage.output_tokens", "cost_usd", "latency_ms", "category", "confidence", "cache_read_tokens", "env"]
PRICE = {"claude-haiku-4-5": (0.80, 4.00), "claude-sonnet-4-6": (3.00, 15.00)}   # $/1M (input, output), illustrative
def build_trace(user_id, model, version, in_tok, out_tok, cache_read, category, confidence):
    pin, pout = PRICE[model]
    cost = (in_tok * pin + out_tok * pout) / 1e6
    return {"trace_id": "tr_" + user_id, "gen_ai.request.model": model, "prompt_version": version, "user_id": user_id,
            "gen_ai.usage.input_tokens": in_tok, "gen_ai.usage.output_tokens": out_tok, "cost_usd": round(cost, 6),
            "latency_ms": 420, "category": category, "confidence": confidence, "cache_read_tokens": cache_read, "env": "prod"}
tr = build_trace("user_8842", "claude-haiku-4-5", "v1.1.0", 1850, 12, 1800, "refund", 0.94)
missing = [f for f in REQUIRED if f not in tr]
print("trace field check: {}/{} required fields present, missing: {}".format(
      len([f for f in REQUIRED if f in tr]), len(REQUIRED), missing or "none - complete"))
for k in ["gen_ai.request.model", "prompt_version", "gen_ai.usage.input_tokens", "cache_read_tokens", "cost_usd", "category"]:
    print("  {:<28} {}".format(k, tr[k]))
print()
print("No trace, no LLMOps: this one record is what the dashboard, the A/B metrics, the cost report, and drift detection ALL read.")

# Output:
# trace field check: 12/12 required fields present, missing: none - complete
#   gen_ai.request.model         claude-haiku-4-5
#   prompt_version               v1.1.0
#   gen_ai.usage.input_tokens    1850
#   cache_read_tokens            1800
#   cost_usd                     0.001528
#   category                     refund
#
# No trace, no LLMOps: this one record is what the dashboard, the A/B metrics, the cost report, and drift detection ALL read.

- The trace record has **twelve required fields** following the OpenTelemetry GenAI conventions (`gen_ai.request.model`, `gen_ai.usage.*`).
- The **cost** is computed from the token counts and the model’s price — here a Haiku call with a cached prefix.
- All twelve fields are present; drop one and a downstream reader breaks.
- This one record feeds the dashboard, the A/B metrics, the cost report, and drift detection — no trace, no LLMOps.

#### 💡 What just happened

⚡ What just happened?Every call emits one twelve-field trace record (OTel GenAI conventions) that the dashboard, A/B metrics, cost report, and drift detection all read; skipping it blinds every downstream capability. Tradeoff: one wrapper on every call, for total observability. Next: the eval harness that turns traces into a ship/no-ship signal.

- A trace record whose twelve fields fill in with a running cost
- Drop a field and a downstream reader (dashboard / A-B / cost / drift) breaks

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: The Eval Harness — Golden Set + Judge

### The Eval Harness — Golden Set + Judge

Run the hand-labeled golden set against a prompt version, grade each case with a multi-dim LLM-as-judge (correctness + format + calibration, different model family), and aggregate the pass rate. The candidate (few-shot) lifts the hard, ambiguous, and multilingual cases - exactly what the few-shot examples target.

The eval harness is what lets you say a prompt change is *better*, not just *different*. It has two pieces. The **golden set** is a hand-labeled list of `(review, expected, difficulty)` cases — at least fifty in the real project, a representative ten-case slice here — deliberately including the hard cases (ambiguous, multilingual, multi-issue) where a weak prompt fails. The **judge** grades each output on a discrete multi-dimension rubric (correctness, format, calibration; pass = all dimensions at least one) using a *stronger, different-family model* so it does not over-reward the producer’s style. Run the set against each version and the story is clear: the baseline aces the easy cases and misses the hard slice, and the few-shot candidate lifts most of the hard cases — because the few-shot examples were chosen to target exactly those failures. The block runs the slice against both versions, keyless; the full fifty-row set lands near eighty and ninety-two percent.

> 📝 **Analogy**
>
> The golden set is a **standardized exam with an answer key**. Every prompt version sits the *same* paper, each question has a known correct answer, and the paper deliberately includes the tricky questions that separate a strong candidate from a lucky one. An exam of only easy questions where everyone scores full marks teaches you nothing — which is exactly why an LLM-synthesized, all-easy golden set is worthless. The judge is the second examiner marking to a written rubric, brought in from outside so they do not just reward their own students’ style.

The baseline v1.0.0 aces the easy cases but fails the hard slice; the few-shot v1.1.0 fixes most of the hard cases. Why does the few-shot version specifically help the hard slice?

**📝 `05_eval_harness.py`** - *runnable*

In [ ]:
# THE EVAL HARNESS: run the golden set against a prompt version, grade each with the multi-dim judge, aggregate the pass rate.
GOLDEN = [   # a representative 10-case slice of the 50-row hand-labeled set (5 easy, 5 hard); the full set lands ~80% / ~92%
    ("Refund never arrived", "refund", "easy"), ("Where is my order?", "delivery", "easy"),
    ("Cold and soggy fries", "food_quality", "easy"), ("The driver was rude", "service", "easy"),
    ("Great food, fast delivery", "other", "easy"), ("Charged twice, still no food", "refund", "hard"),
    ("Two hours late and stone cold", "delivery", "hard"), ("Llego frio y muy tarde", "food_quality", "hard"),
    ("Driver polite but the meal was inedible", "food_quality", "hard"), ("The app keeps crashing on checkout", "other", "hard")]
# deterministic predicted category per version; v1.0.0 (no few-shot) misses 2 hard cases, v1.1.0 (few-shot) fixes 1 of them
PRED = {"v1.0.0": ["refund", "delivery", "food_quality", "service", "other", "refund", "delivery", "delivery", "service", "other"],
        "v1.1.0": ["refund", "delivery", "food_quality", "service", "other", "refund", "delivery", "food_quality", "service", "other"]}
def run_golden(version):                                        # judge grades correctness (+ format + calibration); here correctness drives it
    passed = sum(1 for (rev, exp, diff), pred in zip(GOLDEN, PRED[version]) if pred == exp)
    hard_pass = sum(1 for (rev, exp, diff), pred in zip(GOLDEN, PRED[version]) if pred == exp and diff == "hard")
    return passed, len(GOLDEN), hard_pass
for v in ["v1.0.0", "v1.1.0"]:
    p, n, hp = run_golden(v)
    print("{}: {}/{} = {:.0%} pass   (hard slice {}/5)".format(v, p, n, p / n, hp))
print()
print("The candidate (few-shot) lifts the hard, ambiguous, and multilingual cases - exactly what the few-shot examples target.")
print("The judge scores correctness + format + calibration on a 0-2 rubric with a DIFFERENT-family model; pass = all dims >= 1.")

# Output:
# v1.0.0: 8/10 = 80% pass   (hard slice 3/5)
# v1.1.0: 9/10 = 90% pass   (hard slice 4/5)
#
# The candidate (few-shot) lifts the hard, ambiguous, and multilingual cases - exactly what the few-shot examples target.
# The judge scores correctness + format + calibration on a 0-2 rubric with a DIFFERENT-family model; pass = all dims >= 1.

- The golden set is hand-labeled `(review, expected, difficulty)` cases; a representative slice runs against both versions.
- `v1.0.0` passes the easy cases but misses the hard slice; `v1.1.0` (few-shot) lifts most of the hard cases.
- The judge grades correctness + format + calibration on a discrete **0-2 rubric** with a different-family model; pass = all dimensions at least one.
- This ten-case slice shows the direction; the full fifty-row set lands near eighty percent for the baseline and ninety-two for the candidate.

#### 💡 What just happened

⚡ What just happened?The eval harness runs a hand-labeled golden set against each version and grades with a multi-dim different-family judge; the few-shot candidate lifts exactly the hard cases its examples target. Tradeoff: hand-labeling effort, for an honest better-or-worse signal. Next: the gate that turns that signal into a merge decision.

- A grid of ten golden cases lights green on pass, red on fail
- A champion-vs-candidate pass-rate pair; the candidate lifts the hard slice

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: The CI Gate — Block the Bad Merge

### The CI Gate — Block the Bad Merge

On every PR touching prompts/, the golden set runs and the gate BLOCKS the merge unless three checks pass: a pass-rate FLOOR (ninety percent), a max REGRESSION (two points vs main), and a COST ceiling (thirty percent). A green eval suite is the ship gate - eval-driven development - and this is the single practice most teams have not reached.

The gate is what turns your eval from a script you run by hand into a wall that stops bad changes automatically — and it is the practice that most separates your repo from a demo. A GitHub Actions workflow fires on every pull request that touches `prompts/`, runs the golden set on the candidate versus `main`, and calls `gate.py`, which **exits non-zero (blocking the merge)** unless three checks pass: the candidate clears an absolute **pass-rate floor**, it is not more than a couple of points **below main**, and it does not blow up the **cost** per request. It posts a delta-table comment either way. The block runs three scenarios: the real candidate that *just* clears the floor and merges, a typo’d prompt that is blocked on the floor, and a slow expensive rewrite that passes on quality but is blocked on cost. That last one is the point — a quality win cannot smuggle in a budget blow-up. The block runs the gate, keyless; the workflow that drives it is the illustrative YAML.

> ℹ️ **Info**
>
> ⚙️ The workflow that runs the gate# .github/workflows/llm-eval.yml - the CI eval gate (runs on every PR touching prompts/)
> name: llm-eval
> on:
>   pull_request:
>     paths: ["prompts/**"]
> jobs:
>   eval:
>     runs-on: ubuntu-latest
>     steps:
>       - uses: actions/checkout@v4
>       - run: pip install -r requirements.txt
>       - name: Run golden set + gate
>         env:
>           ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}   # judge needs the key even with OIDC
>         run: python -m eval.gate --candidate ${{ github.head_ref }} --base main
>       # gate.py exits non-zero (blocking the merge) if the candidate falls below the pass-rate floor
>       # OR regresses past the limit vs main OR exceeds the cost ceiling, and posts a delta comment on the PR.

> 👮 **Analogy**
>
> The CI gate is a **bouncer with a three-item checklist**. A good bouncer checks the ID is valid (the floor — are you good enough on your own?), that you are not worse than the regulars already inside (the regression — worse than main?), and that you can cover your tab (the cost ceiling). Fail *any one* and you do not get in, no matter how good the other two look. The gate is that door on your merges: three independent checks, and a single failure blocks the pull request with a comment naming exactly what failed.

A candidate prompt scores well above the floor but costs about fifty percent more per request than main. What does the three-part gate do?

**📝 `06_ci_gate.py`** - *runnable*

In [ ]:
# THE CI EVAL GATE: on every PR touching prompts/, run the golden set and BLOCK the merge unless THREE checks pass.
FLOOR, MAX_REGRESSION, MAX_COST_INCREASE = 0.90, 0.02, 0.30    # floor 90%, max 2pp regression vs main, max +30% cost
def gate(candidate_pass, main_pass, cost_ratio):
    checks = {"floor >= 90%": candidate_pass >= FLOOR,
              "regression <= 2pp vs main": candidate_pass >= main_pass - MAX_REGRESSION,
              "cost <= +30%": cost_ratio <= 1 + MAX_COST_INCREASE}
    return all(checks.values()), checks
SCENARIOS = [("candidate v1.1.0", 0.90, 0.80, 1.00), ("a typo'd bad prompt", 0.83, 0.80, 1.00), ("a slow expensive rewrite", 0.93, 0.80, 1.55)]
for name, cand, main, cost in SCENARIOS:
    ok, checks = gate(cand, main, cost)
    reason = "" if ok else "  <- " + ", ".join(k for k, v in checks.items() if not v) + " failed"
    print("{:<24} pass {:.0%}, cost x{:.2f}  ->  {}{}".format(name, cand, cost, "MERGE" if ok else "BLOCK", reason))
print()
print("A green eval suite is the ship gate (eval-driven development). The candidate JUST clears the 90% floor; the bad PR and the")
print("expensive rewrite are each blocked, with the failed check named as a PR comment. This is the gate that makes you employable.")

# Output:
# candidate v1.1.0         pass 90%, cost x1.00  ->  MERGE
# a typo'd bad prompt      pass 83%, cost x1.00  ->  BLOCK  <- floor >= 90% failed
# a slow expensive rewrite pass 93%, cost x1.55  ->  BLOCK  <- cost <= +30% failed
#
# A green eval suite is the ship gate (eval-driven development). The candidate JUST clears the 90% floor; the bad PR and the
# expensive rewrite are each blocked, with the failed check named as a PR comment. This is the gate that makes you employable.

- Every PR touching `prompts/` runs the golden set; the gate returns MERGE or BLOCK.
- Three checks: a pass-rate **floor**, a max **regression** versus main, and a **cost** ceiling — failing any one blocks.
- The candidate *just* clears the floor and merges; a typo’d prompt is blocked on the floor; an expensive rewrite is blocked on cost.
- A green eval suite is the ship gate (eval-driven development); the failed check is posted as a PR comment.

#### 💡 What just happened

⚡ What just happened?The CI gate runs the golden set on every prompt PR and blocks the merge unless a floor, a regression limit, and a cost ceiling all pass; a quality win cannot smuggle in a cost blow-up. Tradeoff: some CI minutes per PR, for never shipping a silent regression. Next: projecting the cost and scoring yourself before you ship.

- Three PRs pass through a three-check gate (floor, regression, cost)
- One is blocked and gets a PR comment naming the failed check

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Ship It — Project the Cost, Score the Checklist

### Ship It — Project the Cost, Score the Checklist

Project the monthly cost at 1x / 10x / 100x traffic (routing + caching applied), score the go-live checklist, and deploy. A senior reviewer should read the repo in five minutes and see production discipline most teams never reach. Then open-source it, write it up, and reference it in interviews.

The last step is turning a working pipeline into a shippable, defensible artifact. First, **project the cost**: take the blended per-request cost from your traces (after routing sends cheap traffic to Haiku and caching discounts the prefix) and multiply out to monthly burn at today’s traffic, ten times, and a hundred times — the number a reviewer will ask for. Then run the **go-live checklist**: the endpoint returns the contract, every call is traced, two versions are A/B-routed, the golden set and judge exist, the CI gate blocks a real regression, routing and caching are on and visible in traces, the dashboard is deployed with a README cost analysis, and no secrets are committed — and, because this repo goes public, the committed golden set and any stored traces are **PII-scrubbed**: real names, addresses, phone numbers, and order or ticket IDs are anonymized or synthesized before the repo ships. Deploy to Render, Fly, or Railway, and put the URL in the README. Then **use it**: open-source it under a permissive license, write eight hundred words on how you built it, and walk a senior interviewer through the architecture and the gate blocking a regression. The block projects the cost and scores the checklist, keyless.

> 🛸️ **Analogy**
>
> Shipping is the **pre-flight checklist**. A pilot does not eyeball the plane and take off on vibes — they run a fixed list, and every item is green before the wheels leave the ground, because the cost of a missed item is catastrophic and invisible until it is not. Your go-live checklist is that list: the cost projection, the traced calls, the gate that fires, the deployed dashboard, the absent secrets. Eight of eight green, and a stranger who lands on your repo sees a pilot, not a passenger.

You want a senior reviewer to see production discipline in five minutes. Of these, which is the strongest single signal?

**📝 `07_ship_it.py`** - *runnable*

In [ ]:
# SHIP IT: project the monthly cost at 1x / 10x / 100x traffic (routing + caching applied) and score the go-live checklist.
reqs_per_day = 5000
COST_PER_REQ = 0.00042        # $ blended per request AFTER length routing (cheap traffic -> Haiku) + prompt caching (from the traces)
def monthly(mult): return reqs_per_day * 30 * COST_PER_REQ * mult
print("Cost projection (blended, after routing + caching):")
for mult, label in [(1, "today"), (10, "10x"), (100, "100x")]:
    print("  {:<6} {:>12,} req/mo  ->  ${:>9.2f}/mo".format(label, reqs_per_day * 30 * mult, monthly(mult)))
CHECKLIST = [("endpoint returns the JSON contract", True), ("every call traced (12 fields)", True),
             ("2 prompt versions, deterministic A/B", True), ("50-row golden set + multi-dim judge", True),
             ("CI gate blocks a real regression", True), ("routing + caching on, verified in traces", True),
             ("dashboard deployed + README cost analysis", True), ("no committed secrets; golden set + traces PII-scrubbed", True)]
done = sum(1 for _, ok in CHECKLIST if ok)
print()
print("Go-live checklist: {}/{} green.".format(done, len(CHECKLIST)))
print("A senior reviewer reads this repo in five minutes and sees production discipline most teams never reach. Ship it.")

# Output:
# Cost projection (blended, after routing + caching):
#   today       150,000 req/mo  ->  $    63.00/mo
#   10x       1,500,000 req/mo  ->  $   630.00/mo
#   100x     15,000,000 req/mo  ->  $  6300.00/mo
#
# Go-live checklist: 8/8 green.
# A senior reviewer reads this repo in five minutes and sees production discipline most teams never reach. Ship it.

- The **cost projection** scales the blended per-request cost (after routing + caching) to monthly burn at 1x, 10x, and 100x traffic.
- The **go-live checklist** scores the eight things a senior reviewer looks for — contract, trace, A/B, eval, gate, cost levers, dashboard, no secrets.
- All eight are green: the repo shows production discipline most teams never reach.
- Ship it, open-source it, write it up, and walk an interviewer through the architecture and the gate blocking a regression.

#### 💡 What just happened

⚡ What just happened?Shipping means projecting the cost at 10x/100x, scoring the eight-item go-live checklist, deploying, and then open-sourcing and writing it up. Tradeoff: a final polish pass, for an artifact a stranger reads in five minutes. That closes the loop: route it, register it, trace it, eval it, gate it, and ship it.

- A cost-projection chart at 1x / 10x / 100x traffic
- A go-live checklist that turns green, eight of eight

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture — one operated feature
> The capstone is two wired paths and a ship gate. The **request path** (Steps 1-4): a request is routed to a version and a model (Step 2), rendered from the registry with a cacheable prefix (Step 3), and answered by an instrumented call that writes the one trace every reader depends on (Step 4). The **CI path** (Steps 5-6): every prompt PR runs the hand-labeled golden set through a different-family judge (Step 5) and is blocked unless the three-check gate passes (Step 6). And **ship** (Step 7): project the cost, score the checklist, deploy, and open-source it. Ask two questions of any LLM feature you build from here: can you see what every request actually did, and does every prompt change have to earn its merge?

**📦 **Self-assessment rubric****

| Area | Pts | Pass criteria |
|---|---|---|
| Working endpoint | 10 | POST /classifyreturns category + confidence + version + model; handles bad input gracefully. |
| Instrumentation | 15 | Every call writes a trace with the full field set; browsable, searchable by user_id and prompt_version. |
| Prompt versioning | 10 | Two or more YAML versions, SemVer filenames, Jinja withStrictUndefined. |
| A/B routing | 10 | Hash-deterministic bucketing on user_id; the split shows on every trace and the dashboard. |
| Golden set | 15 | Fifty or more hand-labeled rows, mixed difficulty, incl. ambiguous + multilingual + edge cases. |
| LLM-as-judge | 10 | Multi-dim rubric, a different model family than the producer, judge prompt versioned. |
| CI gate | 15 | Runs on every prompt PR; floor + regression + cost; posts a comment; verified to block a real regression. |
| Cost optimization | 5 | Model routing + prompt caching on;cache_read_tokensvisible in traces. |
| Dashboard | 5 | Traffic by version, cost / latency / pass-rate, recent traces; deployed and reachable. |
| README + polish | 5 | Architecture diagram, how-to-run, eval results, cost analysis, demo link, no committed secrets, and a PII-scrubbed golden set (real reviews anonymized before the repo goes public). |

> 📦 **Info**
>
> ➡️ Where this goes nextYou started Module 14 able to *describe* LLMOps; you are finishing it able to *do* it, in a repo that proves it. The next move is scale: a deep, multi-feature product build — an agentic RAG system with its own retrieval, multi-region serving, and governance — is the separate **Project Course** this capstone is the on-ramp to. Within this course, **Module 15 (Ethics & Responsible AI)** is the non-engineering layer of shipping GenAI. When you ship the capstone, open-source it and reference it: the primary tools are [Langfuse](https://github.com/langfuse/langfuse) for tracing, [FastAPI](https://github.com/fastapi/fastapi) for the endpoint, and the [OpenTelemetry GenAI conventions](https://github.com/open-telemetry/semantic-conventions/tree/main/docs/gen-ai) for the trace field set.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **LLMOps Capstone— Ship a Production Pipeline**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-14_6.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-14.6.html` - regenerate this notebook whenever the source HTML is updated.*
